In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (confusion_matrix, roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score)
import openpyxl

# ================================
# 1. Load Data and Select Predictors
# ================================
data = pd.read_excel('Multiclass data.xlsx')
#X = data.drop(columns=['STATUS', 'TMTA', 'TMTB'])
X = data.drop(columns=['STATUS', 'LPARSOPER', 'LTEMPPOLE', 'LROACG', 'LSPLOBE', 'LFRONTALPOLE', 'RCUNEUS', 'RPOSTCENTG', 'RSMG', 'RTTG'])
#X = data.drop(columns=['STATUS'])




# Select the features you want (instead of dropping)
# desired_features = ['LPARSOPER', 'LTEMPPOLE', 'LROACG']  # Replace with your selected features
# X = data[desired_features]


y = data['STATUS']

# Initialize Excel writer
excel_writer = pd.ExcelWriter("Multiclass_Test_Metrics.xlsx", engine="openpyxl")

# Number of runs
n_runs = 5

# Define parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 18],
    'max_depth': [2, 4, 6, 8],
    'min_samples_split': [2, 4, 6, 8],
    'min_samples_leaf': [1, 2, 3]
}

# Initialize storage
summary_results = []
best_params_results = []

# ================================
# 2. Run Multiclass Random Forest
# ================================
for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}")

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=np.random.randint(1,1000)
    )

    # Scale features
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # GridSearchCV for best hyperparameters
    model = RandomForestClassifier(random_state=run)
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='roc_auc_ovr', n_jobs=-1)
    grid_search.fit(X_train_scaled, y_train)
    best_model = grid_search.best_estimator_

    # Store best parameters
    best_params = grid_search.best_params_
    best_params_results.append({"Run": run, **best_params})
    print(f"Best Parameters for Run {run}: {best_params}")

    # ================================
    # 3. Cross-Validation Metrics
    # ================================
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=run)
    cv_results = []
    fold = 1
    for train_idx, val_idx in skf.split(X_train_scaled, y_train):
        X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        best_model.fit(X_fold_train, y_fold_train)

        y_fold_pred = best_model.predict(X_fold_val)
        y_fold_proba = best_model.predict_proba(X_fold_val)

        tn, fp, fn, tp = 0, 0, 0, 0  # Dummy placeholders for multiclass; metrics below are macro
        accuracy = accuracy_score(y_fold_val, y_fold_pred)
        precision = precision_score(y_fold_val, y_fold_pred, average='macro')
        recall = recall_score(y_fold_val, y_fold_pred, average='macro')
        f1 = f1_score(y_fold_val, y_fold_pred, average='macro')
        auc_roc = roc_auc_score(pd.get_dummies(y_fold_val), y_fold_proba, multi_class='ovr')
        auc_pr = average_precision_score(pd.get_dummies(y_fold_val), y_fold_proba, average='macro')

        cv_results.append([accuracy, precision, recall, f1, auc_roc, auc_pr])
        print(f"Run {run} - Fold {fold}: Accuracy={accuracy:.3f}, F1={f1:.3f}")
        fold += 1

    cv_results_df = pd.DataFrame(cv_results, columns=[
        "Accuracy", "Precision", "Recall", "F1", "AUC-ROC", "AUC-PR"
    ])
    cv_results_df.to_excel(excel_writer, sheet_name=f"Run_{run}_CV", index=False)

    # ================================
    # 4. Test Metrics
    # ================================
    best_model.fit(X_train_scaled, y_train)
    y_test_pred = best_model.predict(X_test_scaled)
    y_test_proba = best_model.predict_proba(X_test_scaled)

    accuracy = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred, average='macro')
    recall = recall_score(y_test, y_test_pred, average='macro')
    f1 = f1_score(y_test, y_test_pred, average='macro')
    auc_roc = roc_auc_score(pd.get_dummies(y_test), y_test_proba, multi_class='ovr')
    auc_pr = average_precision_score(pd.get_dummies(y_test), y_test_proba, average='macro')

    summary_results.append([run, accuracy, precision, recall, f1, auc_roc, auc_pr])

    test_results_df = pd.DataFrame([[accuracy, precision, recall, f1, auc_roc, auc_pr]],
                                   columns=["Accuracy", "Precision", "Recall", "F1", "AUC-ROC", "AUC-PR"])
    test_results_df.to_excel(excel_writer, sheet_name=f"Run_{run}_Test", index=False)

    # ================================
    # 5. Confusion Matrix
    # ================================
    conf_matrix = confusion_matrix(y_test, y_test_pred)
    conf_matrix_df = pd.DataFrame(
        conf_matrix,
        index=[f"Actual_{c}" for c in np.unique(y)],
        columns=[f"Predicted_{c}" for c in np.unique(y)]
    )
    conf_matrix_df.to_excel(excel_writer, sheet_name=f"Run_{run}_ConfMatrix")

# ================================
# 6. Save Summary Metrics
# ================================
summary_df = pd.DataFrame(summary_results, columns=[
    "Run", "Accuracy", "Precision", "Recall", "F1", "AUC-ROC", "AUC-PR"
])
summary_df.to_excel(excel_writer, sheet_name="Summary", index=False)

# ================================
# 7. Save Best Parameters
# ================================
best_params_df = pd.DataFrame(best_params_results)
best_params_df.to_excel(excel_writer, sheet_name="Best_Parameters", index=False)

# ================================
# 8. Finalize Excel
# ================================
excel_writer.close()
print("Evaluation metrics, confusion matrices, and best parameters saved successfully!")


Run 1/5
Best Parameters for Run 1: {'max_depth': 4, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 5}
Run 1 - Fold 1: Accuracy=0.500, F1=0.461
Run 1 - Fold 2: Accuracy=0.375, F1=0.374
Run 1 - Fold 3: Accuracy=0.375, F1=0.325
Run 1 - Fold 4: Accuracy=0.438, F1=0.401
Run 1 - Fold 5: Accuracy=0.531, F1=0.537
Run 2/5
Best Parameters for Run 2: {'max_depth': 2, 'min_samples_leaf': 3, 'min_samples_split': 2, 'n_estimators': 6}
Run 2 - Fold 1: Accuracy=0.469, F1=0.421
Run 2 - Fold 2: Accuracy=0.406, F1=0.368
Run 2 - Fold 3: Accuracy=0.531, F1=0.463
Run 2 - Fold 4: Accuracy=0.438, F1=0.417
Run 2 - Fold 5: Accuracy=0.312, F1=0.295
Run 3/5
Best Parameters for Run 3: {'max_depth': 2, 'min_samples_leaf': 3, 'min_samples_split': 2, 'n_estimators': 7}
Run 3 - Fold 1: Accuracy=0.375, F1=0.352
Run 3 - Fold 2: Accuracy=0.438, F1=0.411
Run 3 - Fold 3: Accuracy=0.406, F1=0.413
Run 3 - Fold 4: Accuracy=0.562, F1=0.492
Run 3 - Fold 5: Accuracy=0.469, F1=0.466
Run 4/5
Best Parameters for Run

/home/prasad/miniconda3/envs/ntkenv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/prasad/miniconda3/envs/ntkenv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/prasad/miniconda3/envs/ntkenv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri